# Model Architecture - ChatKasir v0
- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## 1. Imports Library

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# verifikasi versi
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

# check GPU yang tersedia
print(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow: 2.20.0
NumPy: 2.0.2
Pandas: 2.2.2
GPU tersedia: True


## 2. Eksplorasi Awal Dataset Food

In [2]:
# load dataset
url_food_utama = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"

df_food = pd.read_csv(url_food_utama)

# cek ukuran dataset food
print(df_food.shape)

(18558, 1)


In [3]:
# cek 10 baris awal data
df_food.head(10)

,name
0,abon
1,abon ayam
2,abon burger
3,abon cheese burger
4,abon goreng ayam
5,abon goreng sapi
6,abon gulung
7,abon haruwan
8,abon keju
9,abon pedas


In [4]:
# cek tipe data per kolom
df_food.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18558 entries, 0 to 18557
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    18558 non-null  object
dtypes: object(1)
memory usage: 145.1+ KB


In [5]:
# cek nilai kosong
df_food.isnull().sum()


,0
name,0


## 3. Eksplorasi Awal Dataset Slang

In [6]:
# load dataset
url_slang_utama = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"

df_slang = pd.read_csv(url_slang_utama)

# cek ukuran dataset slang
print(df_slang.shape)

(1231, 2)


In [7]:
# cek 10 baris awal data
df_slang.head(10)

,slang,formal
0,aa,kakak
1,abanggg,abang
2,abanggku,abangku
3,abeees,habis
4,abes,habis
5,abez,habis
6,abg,abang
7,abgnya,abangnya
8,abgnyaah,abangnya
9,abiez,habis


In [8]:
# cek tipe data per kolom
df_slang.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   slang   1231 non-null   object
 1   formal  1231 non-null   object
dtypes: object(2)
memory usage: 19.4+ KB


In [9]:
# cek nilai kosong
df_slang.isnull().sum()

,0
slang,0
formal,0


## 4. Eksplorasi Awal Dataset Sintetis 10k

In [10]:
# load dataset
url_sintetis_10k = "https://drive.google.com/uc?id=1Sh4Fx-coZPe14xZP_7XE9QoB5wiKM141"

df_sintetis = pd.read_csv(url_sintetis_10k)

# cek ukuran dataset sintetis
print(df_sintetis.shape)

(10050, 5)


In [11]:
# cek 10 baris awal data
df_sintetis.head(10)

,input_text,product,quantity,price_satuan,pattern
0,boleh pesan 9 crepes beef burger dong [SEP] ok...,crepes beef burger,9,68000,1
1,beli 10 baileys latte [SEP] baileys latte 52 r...,baileys latte,10,52000,2
2,pesan 2 kailan daging ayam dong kak [SEP] siap...,kailan daging ayam,2,12000,1
3,10 nasi goreng ati ampela dong [SEP] oke nasi ...,nasi goreng ati ampela,10,60000,1
4,kk mau pesen 3 nasi goreng original nyam nyam ...,nasi goreng original nyam nyam,3,13000,1
5,pesan 7 lumpsang meses susu dong [SEP] noted k...,lumpsang meses susu,7,-1,1
6,boleh pesan 8 nasi goreng dendeng lemak sapi k...,nasi goreng dendeng lemak sapi,8,35000,1
7,minta 4 mie ayam telor dong [SEP] uke kak mie ...,mie ayam telor,4,68000,3
8,10 ayam saos mentega [SEP] ayam saos mentega 3...,ayam saos mentega,10,31000,2
9,mau 9 donut paha ayam ya kak [SEP] oke donut p...,donut paha ayam,9,69000,2


In [12]:
# cek tipe data per kolom
df_sintetis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10050 entries, 0 to 10049
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   input_text    10050 non-null  object
 1   product       10050 non-null  object
 2   quantity      10050 non-null  int64 
 3   price_satuan  10050 non-null  int64 
 4   pattern       10050 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 392.7+ KB


In [13]:
# cek nilai kosong
df_sintetis.isnull().sum()

,0
input_text,0
product,0
quantity,0
price_satuan,0
pattern,0


In [14]:
# cek distribusi kolom pattern
# memastikan proporsi ketiga pola sudah sesuai kesepakatan
print("Distribusi pattern:")
print(df_sintetis['pattern'].value_counts())
print(f"\nProporsi pattern:")
print(df_sintetis['pattern'].value_counts(normalize=True).round(4))

Distribusi pattern:
pattern
2    3903
1    3896
3    2251
Name: count, dtype: int64

Proporsi pattern:
pattern
2    0.3884
1    0.3877
3    0.2240
Name: proportion, dtype: float64


In [15]:
# cek baris tanpa harga (price_satuan = -1)
n_null_price = (df_sintetis['price_satuan'] == -1).sum()
total = len(df_sintetis)

print(f"Baris dengan price_satuan = -1 : {n_null_price} ({n_null_price/total:.1%})")
print(f"Baris dengan price_satuan valid: {total - n_null_price} ({(total-n_null_price)/total:.1%})")

Baris dengan price_satuan = -1 : 1750 (17.4%)
Baris dengan price_satuan valid: 8300 (82.6%)


In [16]:
# temuan penting untuk hyperparameter model
n_produk_unik = df_sintetis['product'].nunique()
panjang_teks = df_sintetis['input_text'].str.split().str.len()

print(f"Jumlah produk unik di dataset sintetis: {n_produk_unik}")
print("Nilai ini menentukan NUM_PRODCUTS di arsitektur model")

print(f"\nPanjang teks (dalam token/kata):")
print(f"Minimum: {panjang_teks.min()}")
print(f"Rata-rata: {panjang_teks.mean():.1f}")
print(f"Maksimum: {panjang_teks.max()}")
print(f"Persen 95: {panjang_teks.quantile(0.95):.0f}")


Jumlah produk unik di dataset sintetis: 500
Nilai ini menentukan NUM_PRODCUTS di arsitektur model

Panjang teks (dalam token/kata):
Minimum: 4
Rata-rata: 16.7
Maksimum: 27
Persen 95: 22


## 5. Arsitektur Model dan Verifikasi

### Hyperparameter Berdasarkan Eksplorasi Dataset

In [17]:
# jumlah kata unik yang dikenali model
VOCAB_SIZE = 10000

# nilai persentil 95 yang didapatkan saat eksplorasi dataset
# nilainya 22, namun menggunakan nilai sedikit di atasnya sebagai buffer
MAX_SEQ_LEN = 30

# dimensi vektor representasi setiap kata
EMBEDDING_DIM = 128

# ukuran hidden layer satu arah LSTM
# output bidirectional = LSTM_UNITS x 2 = 128
LSTM_UNITS = 64

# nilai produk unik dari eksplorasi dataset sintetis
NUM_PRODUCTS = n_produk_unik
print(f"NUM_PRODUCTS (dari dataset): {NUM_PRODUCTS}")

NUM_PRODUCTS (dari dataset): 500


### Custom Loss Function - MaskedPriceLoss

Custom Loss Function untuk output head 'price_satuan'.

Masalah yang diselesaikan:
Dataset ChatKasir memiliki baris dengan price_satuan = -1 (harga tidak disebutkan di chat). Loss function standar MSE akan menghukum model untuk kasus yang tidak memiliki jawaban valid.

Solusi:
Hitung loss HANYA untuk baris yang memiliki harga valid (bukan -1). Baris dengan harga -1 dikecualikan dari perhitungan gradient.

Konsep masking terinspirasi dari Lample et al. (2016) yang mengabaikan token padding saat menghitung loss pada sequence labeling.

In [18]:
class MaskedPriceLoss(tf.keras.losses.Loss):
    def __init__(self, name="masked_price_loss"):
        super().__init__(name=name)
        self.NULL_INDICATOR = -1.0

    def call(self, y_true, y_pred):
        # buat mask 1 untuk baris dengan harga valid
        mask = tf.cast(
            tf.not_equal(y_true, self.NULL_INDICATOR),
            dtype=tf.float32
        )

        # hitung squared error semua baris
        squared_error = tf.square(y_true - y_pred)

        # terapkan mask tanpa baris null
        masked_error = squared_error * mask

        # rata-rata jumlah baris valid saja
        n_valid = tf.maximum(tf.reduce_sum(mask), 1.0)
        return tf.reduce_sum(masked_error) / n_valid

# verifikasi MaskedPriceLoss bisa diintansiasi tanpa error
masked_loss = MaskedPriceLoss()
print(f"MaskedPriceLoss berhasil didefinisikan: {masked_loss.name}")

MaskedPriceLoss berhasil didefinisikan: masked_price_loss


### Mendefinisikan Arsitektur Model

Membangun model ekstraksi entitas ChatKasir.

- Pendekatan: Multi-Output dengan Shared Bidirectional LSTM Encoder.
- Referensi: Lample et al. (2016) - Neural Architectures for NER.

Returns:

tf.keras.Model dengan tiga output:
- product  : softmax (klasifikasi nama produk)
- quantity : relu (regresi jumlah pesanan)
- price_satuan    : relu (regresi harga satuan (dinormalisasi ÷1000))

In [19]:
def build_model(vocab_size=VOCAB_SIZE, max_seq_len=MAX_SEQ_LEN, embedding_dim=EMBEDDING_DIM, lstm_units=LSTM_UNITS, num_products=NUM_PRODUCTS):

    # ===Input===
    inputs = layers.Input(shape=(max_seq_len,), name="input_tokens")

    # ===Shared Encoder===
    # embedding: ubah integer token menjadi vektor bermakna berdimensi 128
    # mask_zero=True token padding (0) diabaikan dalam komputasi LSTM
    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True,
        name="embedding"
    )(inputs)

    # bidirectional LSTM (baca konteks dari kiri ke kanan dan sebaliknya)
    # return_sequences=False ambil hanya state akhir (ringkasan kalimat)
    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=False),
        name="bi_lstm"
    )(x)

    # dropout 30% neuron acak saat training, mencegah overfitting
    x = layers.Dropout(0.3, name="dropout")(x)

    # shared dense sebelum output terpecah ke 3 head
    x = layers.Dense(128, activation="relu", name="shared_dense")(x)

    # ===Output Head===
    # head 1: product
    product_output = layers.Dense(
        num_products, activation="softmax", name="product"
    )(x)

    # head 2: quantity (selalu positif)
    quantity_output = layers.Dense(
        1, activation="relu", name="quantity"
    )(x)

    # head 3: price (dinormalisasi / 1000)
    price_output = layers.Dense(
        1, activation="relu", name="price_satuan"
    )(x)

    # ===Model===
    model = tf.keras.Model(
        inputs=inputs,
        outputs={
            "product": product_output,
            "quantity": quantity_output,
            "price_satuan": price_output
        },
        name="chatkasir_extractor_v0"
    )
    return model

# instansiasi & tampilkan ringkasan arsitektur
model = build_model()
model.summary()

Model: "chatkasir_extractor_v0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_tokens        │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 30, 128)   │  1,280,000 │ input_tokens[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 30)        │          0 │ input_tokens[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bi_lstm             │ (None, 128)       │     98,816 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bi_lstm[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense        │ (None, 128)       │     16,512 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price_satuan        │ (None, 1)         │        129 │ shared_dense[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product (Dense)     │ (None, 500)       │     64,500 │ shared_dense[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantity (Dense)    │ (None, 1)         │        129 │ shared_dense[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,460,086 (5.57 MB)

 Trainable params: 1,460,086 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

### Verifikasi dengan Input Dummy

In [20]:
# simulasikan 1 kalimat dengan panjang MAX_SEQ_LEN
# nilai integer acak antara 1–VOCAB_SIZE (0 = padding, dihindari)
dummy_input = np.random.randint(1, VOCAB_SIZE, size=(1, MAX_SEQ_LEN))
output = model(dummy_input, training=False)

print("Verifikasi Output Shape:")
print(f"Product : {output['product'].shape}") # harusnya (1, NUM_PRODUCTS)
print(f"Quantity : {output['quantity'].shape}")  # harusnya (1, 1)
print(f"Price Satuan : {output['price_satuan'].shape}") # harusnya (1, 1)

print("\nContoh Nilai Prediksi (Belum Dilatih - Nilai Acak):")
top3 = tf.nn.top_k(output['product'][0], k=3)
print(f"Product top-3 probabilitas : {top3.values.numpy()}")
print(f"Quantity prediksi : {output['quantity'][0][0].numpy():.4f}")
print(f"Price Satuan Prediksi : {output['price_satuan'][0][0].numpy():.4f} (x1000 = {output['price_satuan'][0][0].numpy()*1000:.0f} rupiah)")

Verifikasi Output Shape:
Product : (1, 500)
Quantity : (1, 1)
Price Satuan : (1, 1)

Contoh Nilai Prediksi (Belum Dilatih - Nilai Acak):
Product top-3 probabilitas : [0.00202089 0.00201715 0.00201684]
Quantity prediksi : 0.0000
Price Satuan Prediksi : 0.0000 (x1000 = 0 rupiah)


In [21]:
# rencana model compile (implementasi di 02_training.ipynb Minggu 2)
# sel ini tidak dijalankan - hanya dokumentasi keputusan loss function

"""
model.compile(
    optimizer="adam",
    loss={
        # Klasifikasi: label integer, bukan one-hot
        "product": tf.keras.losses.SparseCategoricalCrossentropy(),

        # Regresi quantity: MSE standar, nilai kecil (1-10)
        "quantity": tf.keras.losses.MeanSquaredError(),

        # Regresi harga: custom, abaikan price_satuan = -1
        # price_satuan dinormalisasi ÷1000 sebelum masuk training
        "price_satuan": MaskedPriceLoss(),
    },
    loss_weights={"product": 1.0, "quantity": 1.0, "price_satuan": 1.0}
)
"""

print("Rencana kompilasi terdokumentasi - implementasi di 02_training.ipynb")

Rencana kompilasi terdokumentasi - implementasi di 02_training.ipynb


## Ringkasan Temuan di Minggu 1

### Temuan dari Eksplorasi Dataset
- `food_utama.csv`: 18.558 baris, tidak ada null
- `slang_utama.csv`: 1.231 baris, tidak ada null
- `NUM_PRODUCTS` dikonfirmasi dari dataset sintetis (lihat hasil eksplorasi)
- `MAX_SEQ_LEN` dikonfirmasi dari persentil 95 panjang teks

### Keputusan Arsitektur yang Sudah Diambil
- Encoder: Bidirectional LSTM (Lample et al., 2016)
- Embedding: dilatih dari nol (kosakata domain-spesifik)
- Komponen kustom: MaskedPriceLoss untuk menangani price_satuan = -1

### Pertanyaan untuk Weekly Sync Jumat (24 April)
- DS-1 (Faradi): apakah nilai sentinel -1 sudah diterapkan di dataset sintetis?
- AI-2 (Denny): konfirmasi bahwa price output model dalam rupiah penuh setelah de-normalisasi x1000
- FS-2 (Reihan): apakah API_CONTRACT sudah mencantumkan field `confidence` dan `total`?

### Catatan Teknis Environment
- GPU tidak tersedia di Windows native untuk TensorFlow >= 2.11
- Training Minggu 2-3 disarankan menggunakan Google Colab (GPU gratis)
  agar eksperimen iteratif bisa selesai dalam hitungan menit, bukan jam
- Notebook ini sudah siap dijalankan di Colab - cukup upload atau mount
  Google Drive dan jalankan dari atas